# Ola Driver Attrition Analysis

## Executive Summary
This project analyzes Ola driver attrition using historical driver data. The objective is to identify the key factors influencing attrition and provide actionable business recommendations to improve driver retention.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

## Business Problem
Recruiting and retaining drivers is a major challenge for Ola. High driver attrition increases hiring costs, affects operational efficiency, and impacts business performance.

## Project Objectives
- Understand the dataset
- Clean and preprocess the data
- Perform feature engineering
- Conduct exploratory data analysis
- Identify factors affecting attrition
- Provide business recommendations

## Data Dictionary
| Column | Description |
|---|---|
| Driver_ID | Unique Driver Identifier |
| Age | Driver Age |
| Gender | Driver Gender |
| Income | Monthly Income |
| Quarterly Rating | Driver Performance |
| Total Business Value | Business Generated |
| LastWorkingDate | Driver Exit Date |

# Import Required Libraries

In [ ]:
# Import Required Libraries

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Display Settings
pd.set_option('display.max_columns', None)

print("Libraries Imported Successfully")

# Load Dataset

In [ ]:
# Load Dataset

df = pd.read_csv("../data/ola_driver.csv")
print("Dataset Loaded Successfully")

# Data Understanding

In [ ]:
# Dataset Overview

df.head()

df.tail()

print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

df.columns

df.info()

# Data Types
df.dtypes

In [ ]:
# Statistical Summary
df.describe().T

In [ ]:
# Missing Values
df.isnull().sum()

In [ ]:
# Missing Value Percentage


missing = pd.DataFrame({
    "Missing Values":df.isnull().sum(),
    "Percentage":round(df.isnull().mean()*100,2)
})

missing

In [ ]:
# Duplicate Records
print("Duplicate Rows :",df.duplicated().sum())

In [ ]:
# Unique Drivers
df["Driver_ID"].nunique()

In [ ]:
# Convert Date Columns
date_columns=["MMM-YY","Dateofjoining","LastWorkingDate"]

for col in date_columns:
    df[col]=pd.to_datetime(df[col])

df.info()

# Data Cleaning & Preprocessing

In this section, we will:
- Check and handle missing values
- Convert data types where required
- Analyze duplicate records
- Prepare the dataset for further analysis

In [ ]:
# Missing Values Count

missing = df.isnull().sum().sort_values(ascending=False)

In [ ]:
# Missing Value Visualization
plt.figure(figsize=(10,5))

missing[missing>0].plot(kind='bar')

plt.title("Missing Values in Dataset")
plt.ylabel("Count")
plt.xlabel("Columns")
plt.savefig("../images/missing_value_Dataset.png", dpi=300, bbox_inches="tight")
plt.show()

### Observation

- LastWorkingDate contains missing values.
- Missing values in LastWorkingDate indicate that the driver is still active.
- Other columns contain very few or no missing values.

In [ ]:
# Check Duplicate Driver Records


df["Driver_ID"].value_counts().head(10)

# Driver Count

print("Total Records :",len(df))
print("Unique Drivers :",df["Driver_ID"].nunique())

### Observation

Each driver appears multiple times because the dataset contains monthly records.

In [ ]:
# Target Variable (Attrition)

df["Attrition"] = np.where(df["LastWorkingDate"].isnull(),0,1)

# 0 → Active Driver
# 1 → Left Ola

df["Attrition"].value_counts()

In [ ]:
# Attrition Percentage

attrition = round(df["Attrition"].value_counts(normalize=True)*100,2)

In [ ]:
# Attrition Chart

plt.figure(figsize=(6,4))
sns.countplot(data=df,x="Attrition")
plt.title("Driver Attrition Distribution")
plt.savefig("../images/driver_attrition_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# Feature Engineering

## Overview

Feature engineering is the process of creating meaningful variables from existing data. These features help uncover patterns, improve analysis, and support better business decisions.

In this section, we will:

- Create the Attrition target variable
- Calculate driver tenure
- Track income growth
- Track quarterly rating improvement
- Create driver-level dataset

In [ ]:
# Sort Dataset
df = df.sort_values(["Driver_ID","MMM-YY"])


In [ ]:
# Driver Level Dataset

driver_df = df.groupby("Driver_ID").last().reset_index()

driver_df.head()



In [ ]:
# Shape
print(driver_df.shape)

In [ ]:
# Attrition Distribution

driver_df["Attrition"].value_counts()



In [ ]:
# Attrition %
round(driver_df["Attrition"].value_counts(normalize=True)*100,2)


In [ ]:
# Attrition Graph

plt.figure(figsize=(6,4))
sns.countplot(data=driver_df,x="Attrition")
plt.title("Driver Attrition")
plt.savefig("../images/driver_attrition.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# Attrition Distribution

driver_df["Attrition"].value_counts()

In [ ]:
# Attrition %

round(driver_df["Attrition"].value_counts(normalize=True)*100,2)


## Driver Tenure

Driver tenure represents the total duration (in days) that a driver remained associated with Ola. It is calculated using the Date of Joining and Last Working Date.

For active drivers (where LastWorkingDate is missing), the latest reporting date is considered as the end date.

In [ ]:
# Create End Date

driver_df["End_Date"] = driver_df["LastWorkingDate"]

driver_df["End_Date"] = driver_df["End_Date"].fillna(driver_df["MMM-YY"])

# Calculate Tenure

driver_df["Tenure_Days"] = (
    driver_df["End_Date"] - driver_df["Dateofjoining"]
).dt.days

driver_df[["Driver_ID", "Dateofjoining", "End_Date", "Tenure_Days"]].head()

In [ ]:
# Convert into Months
driver_df["Tenure_Months"] = round(driver_df["Tenure_Days"] / 30, 1)

driver_df[["Tenure_Days", "Tenure_Months"]].head()

### Observation

- Driver tenure has been calculated using the joining date and end date.
- For active drivers, the latest reporting month has been considered as the end date.
- Tenure will help analyze whether new drivers are more likely to leave the company.

In [ ]:
# Driver Tenure Distribution

plt.figure(figsize=(8,5))
sns.histplot(driver_df["Tenure_Months"], bins=25, kde=True)
plt.title("Distribution of Driver Tenure")
plt.xlabel("Tenure (Months)")
plt.ylabel("Number of Drivers")
plt.savefig("../images/distribution_driver_tenure.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# Summary Statistics
driver_df["Tenure_Months"].describe()

## Income Growth Analysis

To analyze salary progression, we compare the first recorded income with the last recorded income for each driver.

A new feature **Income_Increased** is created:

- **1** → Income increased
- **0** → Income remained the same or decreased

This feature helps identify whether income growth influences driver retention.

In [ ]:
# First Income of each driver
first_income = df.groupby("Driver_ID")["Income"].first()

# Last Income of each driver
last_income = df.groupby("Driver_ID")["Income"].last()

# Map values
driver_df["First_Income"] = driver_df["Driver_ID"].map(first_income)
driver_df["Last_Income"] = driver_df["Driver_ID"].map(last_income)

# Income Increased Feature
driver_df["Income_Increased"] = np.where(
    driver_df["Last_Income"] > driver_df["First_Income"], 1, 0
)

driver_df[["Driver_ID", "First_Income", "Last_Income", "Income_Increased"]].head()

### Observation

- Income growth has been tracked for every driver.
- Drivers whose latest income is higher than their initial income are marked as **1**.
- This feature will help determine whether salary growth has an impact on attrition.

## Quarterly Rating Improvement

Driver performance is evaluated using the quarterly rating.

A new feature **Quarterly_Rating_Increased** is created:

- **1** → Rating improved
- **0** → Rating remained the same or decreased

This helps analyze whether performance improvement is associated with better driver retention.

In [ ]:
# First Rating
first_rating = df.groupby("Driver_ID")["Quarterly Rating"].first()

# Last Rating
last_rating = df.groupby("Driver_ID")["Quarterly Rating"].last()

driver_df["First_Rating"] = driver_df["Driver_ID"].map(first_rating)
driver_df["Last_Rating"] = driver_df["Driver_ID"].map(last_rating)

driver_df["Quarterly_Rating_Increased"] = np.where(
    driver_df["Last_Rating"] > driver_df["First_Rating"], 1, 0
)

driver_df[
    [
        "Driver_ID",
        "First_Rating",
        "Last_Rating",
        "Quarterly_Rating_Increased",
    ]
].head()

### Observation

- Quarterly ratings have been compared between the first and last records of each driver.
- Drivers with improved ratings are labeled as **1**.
- This feature will help analyze the relationship between performance improvement and driver attrition.

In [ ]:
# Verify New Features
driver_df[
    [
        "Tenure_Months",
        "Income_Increased",
        "Quarterly_Rating_Increased",
        "Attrition",
    ]
].head()

# Exploratory Data Analysis (EDA)

## Overview

Exploratory Data Analysis (EDA) is performed to understand the distribution of variables, identify trends, discover relationships between features, and uncover factors influencing driver attrition.

The analysis focuses on driver demographics, income, performance, tenure, and business value.


## 1. Driver Attrition Distribution

This visualization shows the proportion of drivers who have left the company versus those who are still active.

plt.figure(figsize=(7,5))

ax = sns.countplot(data=driver_df, x="Attrition")

plt.title("Driver Attrition Distribution", fontsize=14)
plt.xlabel("Attrition (0 = Active, 1 = Left)")
plt.ylabel("Number of Drivers")

for container in ax.containers:
    ax.bar_label(container)

plt.show()

### Observation

- The chart shows the distribution of active and inactive drivers.
- This helps identify whether the dataset is balanced or skewed.
- A high attrition rate indicates the need for effective driver retention strategies.

## 2. Driver Age Distribution

This visualization shows the age distribution of drivers.

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(driver_df["Age"], bins=20, kde=True)

plt.title("Distribution of Driver Age")
plt.xlabel("Age")
plt.ylabel("Number of Drivers")
plt.savefig("../images/distriutbution_driver_age.png", dpi=300, bbox_inches="tight")

plt.show()

### Observation

- Most drivers belong to the middle-age group.
- Extremely young and older drivers are comparatively fewer.

In [ ]:
# Gender Distribution
plt.figure(figsize=(6,5))

ax = sns.countplot(data=driver_df, x="Gender")

plt.title("Gender Distribution")

ax.set_xticklabels(["Male", "Female"])

for container in ax.containers:
    ax.bar_label(container)
plt.savefig("../images/gender_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

### Observation

- The dataset is dominated by male drivers.
- Female driver participation is comparatively lower.

In [ ]:
# Income Distribution
plt.figure(figsize=(8,5))

sns.histplot(driver_df["Income"], bins=30, kde=True)

plt.title("Income Distribution")

plt.xlabel("Monthly Income")
plt.savefig("../images/income_Distribution.png", dpi=300, bbox_inches="tight")

plt.show()

### Observation

- Driver income is positively skewed.
- Most drivers fall within a specific income range, while a few drivers earn significantly higher incomes.

In [ ]:
# Quarterly Rating Distribution
plt.figure(figsize=(7,5))

ax = sns.countplot(data=driver_df, x="Quarterly Rating")

plt.title("Quarterly Rating Distribution")

for container in ax.containers:
    ax.bar_label(container)
plt.savefig("../images/quarterly_rating_distribution.png", dpi=300, bbox_inches="tight")

plt.show()

### Observation

- Most drivers have average quarterly ratings.
- Very high ratings are comparatively less common.

## 6. Attrition by Gender

This analysis examines whether driver attrition differs between male and female drivers.

In [ ]:
plt.figure(figsize=(7,5))

ax = sns.countplot(data=driver_df, x="Gender", hue="Attrition")

plt.title("Attrition by Gender")
plt.xlabel("Gender (0 = Male, 1 = Female)")
plt.ylabel("Number of Drivers")

for container in ax.containers:
    ax.bar_label(container)
plt.savefig("../images/attrition_gender.png", dpi=300, bbox_inches="tight")

plt.show()

### Observation

- Compare attrition rates between male and female drivers.
- If one gender shows a noticeably higher attrition rate, targeted retention strategies can be developed.

## 7. Attrition by Education Level

This analysis evaluates whether educational qualification has any relationship with driver attrition.

In [ ]:
plt.figure(figsize=(8,5))

ax = sns.countplot(data=driver_df,
                   x="Education_Level",
                   hue="Attrition")

plt.title("Attrition by Education Level")

plt.xlabel("Education Level")
plt.ylabel("Drivers")

for container in ax.containers:
    ax.bar_label(container)
plt.savefig("../images/attrition_education_level.png", dpi=300, bbox_inches="tight")

plt.show()

## 8. Attrition vs Income Growth

This analysis examines whether drivers who received an income increase are more likely to remain with the company.

In [ ]:
plt.figure(figsize=(7,5))

ax = sns.countplot(data=driver_df,
                   x="Income_Increased",
                   hue="Attrition")

plt.title("Attrition vs Income Growth")

plt.xlabel("Income Increased")

for container in ax.containers:
    ax.bar_label(container)
plt.savefig("../images/attrition_vs_income_growth.png", dpi=300, bbox_inches="tight")

plt.show()

EDA 9: Attrition by Rating Improvement


In [ ]:

plt.figure(figsize=(7,5))

ax = sns.countplot(
    data=driver_df,
    x="Quarterly_Rating_Increased",
    hue="Attrition"
)

plt.title("Attrition vs Quarterly Rating Improvement")

plt.xlabel("Quarterly Rating Increased")

for container in ax.containers:
    ax.bar_label(container)
plt.savefig("../images/attrition_vs_quarterly_rating_improvement.png", dpi=300, bbox_inches="tight")

plt.show()

## 10. Driver Tenure vs Attrition

This visualization compares the tenure of active and inactive drivers.

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=driver_df,
    x="Attrition",
    y="Tenure_Months"
)

plt.title("Driver Tenure vs Attrition")
plt.savefig("../images/driver_tenure_vs_attrition.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# EDA 11: Income vs Attrition
plt.figure(figsize=(8,5))

sns.boxplot(
    data=driver_df,
    x="Attrition",
    y="Income"
)

plt.title("Income vs Attrition")
plt.savefig("../images/income_vs_attrition.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# EDA 12: Business Value Distribution
plt.figure(figsize=(8,5))

sns.histplot(
    driver_df["Total Business Value"],
    bins=30,
    kde=True
)

plt.title("Total Business Value Distribution")
plt.savefig("../images/total_business_value_distribution.png", dpi=300, bbox_inches="tight")

plt.show()


### Observation

- The heatmap highlights the correlation between numerical features.
- Strong positive correlations indicate variables moving together.
- Negative correlations indicate inverse relationships.
- Attrition-related features can be analyzed to understand key drivers of employee turnover.

# Key Business Insights

Based on the exploratory data analysis, the following insights were identified:

1. Driver attrition is concentrated within specific driver segments.
2. Income growth appears to influence driver retention.
3. Drivers with improved quarterly ratings tend to have better retention.
4. Short-tenure drivers are more likely to leave the company.
5. Business value generated by drivers varies significantly across the workforce.

# Correlation Analysis

## Overview

Correlation analysis helps identify the strength and direction of relationships between numerical variables. It provides valuable insights into how different factors are associated with driver attrition and overall business performance.

In [ ]:
plt.figure(figsize=(12,8))

correlation = driver_df.select_dtypes(include=["number"]).corr()

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")
plt.savefig("../images/correlation.png", dpi=300, bbox_inches="tight")

plt.show()

### Observation

- The heatmap highlights the correlation between numerical features.
- Strong positive correlations indicate variables moving together.
- Negative correlations indicate inverse relationships.
- Attrition-related features can be analyzed to understand key drivers of employee turnover.

# Final Dataset Overview

The dataset has been successfully cleaned and transformed. New features have been engineered to support business analysis and improve the understanding of driver attrition.

In [ ]:
driver_df.head()
driver_df.info()
driver_df.describe().T

# Business Recommendations

Based on the analysis, the following recommendations can help reduce driver attrition:

### 1. Improve Driver Retention
- Identify drivers with shorter tenure and provide onboarding support.

### 2. Performance-Based Incentives
- Reward drivers with consistently high quarterly ratings.

### 3. Salary & Incentive Review
- Offer timely income revisions and performance bonuses.

### 4. Early Attrition Monitoring
- Build an early warning system to identify drivers at risk of leaving.

### 5. Training Programs
- Support low-performing drivers through coaching and mentoring.

### 6. Promotion Opportunities
- Recognize high-performing drivers with career growth opportunities.

# Conclusion

This project analyzed Ola's driver dataset to understand the factors influencing driver attrition.

Key preprocessing tasks included handling missing values, feature engineering, and creating a driver-level dataset. Exploratory Data Analysis revealed patterns related to driver demographics, tenure, income, quarterly ratings, and business value.

The insights generated from this analysis can help Ola design targeted retention strategies, improve driver engagement, and reduce recruitment costs.